# Language Models for Information Retrieval
## Topic 2 · Query likelihood · Extended to BM25

This notebook compares three ways to rank a small document collection: TF-IDF cosine similarity, Dirichlet query likelihood, and BM25. We use a shared implementation in `retrieval.py` so the notebook and Streamlit demo use the same scoring code.

Our question is simple: **which method puts the documents that answer a query near the top?**

**How to run:** install the packages in `requirements.txt`, then run all notebook cells from top to bottom. The collection and implementation are included in this project; no API key or dataset download is needed.


## 1. What we use from Lecture 2 and Lecture 5

| Lecture idea | How we use it |
|---|---|
| Bag of words, Lecture 2 | Count terms in each document. |
| Unigram language model, Lecture 2 | Estimate the probability of each query word under a document model. |
| Log probabilities and smoothing, Lecture 2 | Add log scores and give partial matches usable probabilities. |
| TF-IDF and cosine, Lecture 5 | Build a weighted word-vector baseline. |
| BM25, Lecture 5 | Add saturated term frequency and document-length adjustment. |

The language models use a unigram assumption, so they ignore word order.


## 2. Query likelihood

Let $c(w,d)$ be the count of term $w$ in document $d$, and let $|d|$ be its token count. The maximum-likelihood estimate is

$$P_{MLE}(w\mid d)=\frac{c(w,d)}{|d|}.$$

With the unigram independence assumption, the query score is

$$P(q\mid d)=\prod_{w\in q}P(w\mid d)^{c(w,q)},\qquad
s(d,q)=\sum_{w\in q}c(w,q)\log P(w\mid d).$$

Here the product and sum run over **distinct query terms**. Repeated terms still matter through $c(w,q)$.

Bayes' rule gives $P(d\mid q)\propto P(q\mid d)P(d)$. With equal document priors, we rank by $P(q\mid d)$. This is a modeling assumption, not a guarantee of relevance. Larger log scores are better, so −4 ranks above −8.

Boolean AND retrieval cannot rank partial matches and rejects a document missing any required word. TF-IDF adds weighted matching and cosine similarity. Query likelihood provides a probabilistic way to score the query and makes probability estimation, especially smoothing, explicit. It does not solve all vocabulary or meaning problems.

Background: [Manning et al., Chapter 12](https://nlp.stanford.edu/IR-book/pdf/12lmodel.pdf), building on [Ponte & Croft (1998)](https://ciir.cs.umass.edu/pubfiles/ir-120.pdf).

## 3. Why we smooth

Suppose the query contains `search documents`. A document that contains `search` but never `documents` gets $P(q\mid d)=0$ under MLE. Another document missing `search` also gets zero. We lose the ability to distinguish those partial matches.

Let $P(w\mid C)=c(w,C)/|C|$ be the probability of a word in the whole collection.

**Dirichlet smoothing** adds collection-based pseudo-counts:

$$P_\mu(w\mid d)=\frac{c(w,d)+\mu P(w\mid C)}{|d|+\mu}.$$

Here $\mu>0$ controls the background contribution. The equivalent mixture weight is $\mu/(|d|+\mu)$, so shorter documents receive more smoothing for a fixed $\mu$.

**Jelinek–Mercer (JM)** uses a fixed background weight $\lambda$:

$$P_\lambda(w\mid d)=(1-\lambda)\frac{c(w,d)}{|d|}+\lambda P(w\mid C),\quad 0<\lambda<1.$$

We define $\lambda$ as the **background weight** throughout. Some references reverse this convention.

Lecture 2's add-one method instead gives $(c(w,d)+1)/(|d|+|V|)$ for a unigram model. It uses a uniform vocabulary background. Collection smoothing uses unequal probabilities based on how often terms occur. Smoothing can rescue missing terms in a document only if they occur somewhere in the collection.

Reading: [Zhai & Lafferty (2001)](https://www.cs.cmu.edu/~czhai/paper/sigir2001-smooth.pdf).

In [ ]:
from pathlib import Path
import json
import math
import re
from collections import Counter

import pandas as pd
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer

pd.set_option("display.max_colwidth", 100)
OUTPUT = Path("results")
OUTPUT.mkdir(exist_ok=True)
print("Setup ready. No network connection is needed.")

## 4. Collection, queries, and relevance labels

We wrote **15 short English documents and 5 queries** for teaching. Documents cover IR concepts, with two insurance/parking examples to show a vocabulary problem. The relevance labels below are our own binary judgments based on each query's intended information need. They are not independent benchmark labels.

For Q1, we consider documents about language modeling and probability relevant, including bigrams and perplexity. For Q3, D15 is relevant because it describes web pages entering an index, though D03 gives the more direct crawler explanation. For Q5, the intended need is vehicle insurance coverage, so D13 is relevant and parking document D14 is not.

We declare the labels before comparing methods. Both methods receive identical text and tokenization: lowercase alphabetic words, no stop-word removal, stemming, or phrase features. This keeps the connection to Lecture 2 clear. It also means `model` and `models` remain different terms.

In [ ]:
collection = {'description': 'Handwritten teaching collection. Binary relevance labels are declared by the '
                'author, not benchmark judgments.',
 'documents': {'D01': 'Search engines rank documents for a user query. Ranking helps users '
                      'find relevant information.',
               'D02': 'An inverted index maps each term to documents. The index supports fast '
                      'search over a collection.',
               'D03': 'A web crawler downloads web pages and follows links. Crawling collects '
                      'pages for a search engine.',
               'D04': 'Language models assign probabilities to sequences of words. A unigram '
                      'model uses individual word counts.',
               'D05': 'Query likelihood ranks documents using a document language model. It '
                      'estimates the probability of the query.',
               'D06': 'Smoothing gives probability to unseen words. Dirichlet smoothing '
                      'combines document counts with collection probabilities.',
               'D07': 'A bigram language model predicts a word using the previous word. '
                      'Bigrams preserve local word order.',
               'D08': 'TF IDF weights terms using term frequency and inverse document '
                      'frequency. Cosine similarity compares document and query vectors.',
               'D09': 'BM25 is a ranking function with term frequency saturation and document '
                      'length normalization.',
               'D10': 'Relevance feedback expands a query using relevant documents. Expansion '
                      'can reduce vocabulary mismatch in search.',
               'D11': 'Neural embeddings represent semantic meaning. Dense retrieval can '
                      'match related words with different spellings.',
               'D12': 'Perplexity evaluates how well a language model predicts held out text. '
                      'Lower perplexity means higher average token probability.',
               'D13': 'An automobile insurance policy covers vehicle damage and accident '
                      'liability. Drivers pay a premium for coverage.',
               'D14': 'A car park provides parking spaces near the city market. Visitors '
                      'leave a car while shopping.',
               'D15': 'Web pages contain links and text. An index stores web content so a '
                      'search engine can retrieve pages.'},
 'queries': [{'id': 'Q1',
              'text': 'language model probability',
              'relevant': ['D04', 'D05', 'D07', 'D12']},
             {'id': 'Q2', 'text': 'unseen words smoothing', 'relevant': ['D06']},
             {'id': 'Q3', 'text': 'crawler web pages', 'relevant': ['D03', 'D15']},
             {'id': 'Q4', 'text': 'ranking documents query', 'relevant': ['D01', 'D05']},
             {'id': 'Q5', 'text': 'car coverage', 'relevant': ['D13']}],
 'diagnostic_queries': ['car protection', 'quantum teleportation', 'language quantum']}

documents = collection["documents"]
queries = collection["queries"]
display(pd.DataFrame([{"id": key, "text": value} for key, value in documents.items()]))
display(pd.DataFrame(queries))

## 5. Implementation

The shared implementation in `retrieval.py` counts words once, then supports TF-IDF cosine, MLE, Jelinek–Mercer, Dirichlet smoothing, and BM25. The main comparison below reports only TF-IDF, Dirichlet, and BM25.

For TF-IDF, scikit-learn uses raw term frequency, smoothed IDF, and L2 normalization. For the language models, we sum log probabilities. BM25 uses the lecture's smoothed IDF with default settings $k_1=1.2$ and $b=0.75$.

**Unknown query words:** terms absent from the collection are reported and ignored. A query with no known terms returns no ranking. This is an explicit policy; smoothing cannot infer an unseen synonym.


In [ ]:
from retrieval import RetrievalExperiment, evaluate, tokenize


In [ ]:
experiment = RetrievalExperiment(documents)
print("Documents:", len(documents))
print("Tokens:", experiment.collection_length)
print("Vocabulary:", len(experiment.collection_counts))
print("Average document length:", round(experiment.collection_length / len(documents), 2))
display(pd.DataFrame({"document": experiment.document_ids,
                      "length": [experiment.lengths[d] for d in experiment.document_ids]}))

## 6. One calculation we can explain by hand

We use a separate three-document collection here to keep the arithmetic short:

- A: `search engines rank documents documents`
- B: `search engines index web pages`
- C: `language models rank documents`

There are 14 tokens in total. `search` occurs twice and `documents` occurs three times. For query `search documents`, document C has no `search`, so its MLE query probability is zero. With $\mu=5$, it receives a small background probability instead.

This cell checks the arithmetic directly against our implementation.

In [ ]:
small = RetrievalExperiment({
    "A": "search engines rank documents documents",
    "B": "search engines index web pages",
    "C": "language models rank documents",
})
rows = []
for document_id in small.document_ids:
    p_search = small.term_probability("search", document_id, mu=5)
    p_documents = small.term_probability("documents", document_id, mu=5)
    rows.append({"document": document_id, "P(search|d)": p_search,
                 "P(documents|d)": p_documents,
                 "P(query|d)": p_search * p_documents,
                 "log score": math.log(p_search) + math.log(p_documents)})
worked = pd.DataFrame(rows)
display(worked.round(6))
assert small.collection_length == 14
assert math.isclose(rows[2]["P(search|d)"], (5 * 2 / 14) / 9)
assert math.isclose(rows[2]["P(documents|d)"], (1 + 5 * 3 / 14) / 9)
assert small.rank("search documents", "mle")[0][2][1] == -math.inf

## 7. Main experiment

We compare **TF-IDF cosine**, **Dirichlet query likelihood with $\mu=20$**, and **BM25 with $k_1=1.2$ and $b=0.75$**. These are illustrative settings, not values tuned on a held-out set.

The output includes scores and ranks. Scores from different methods have different meanings, so we compare ranked lists with the same two relevance measures.


In [ ]:
methods = {
    "TF-IDF": "tfidf",
    "Dirichlet (20)": "dirichlet",
    "BM25 (1.2, 0.75)": "bm25",
}
ranking_rows = []
for query in queries:
    for label, method in methods.items():
        ranking, unknown = experiment.rank(query["text"], method=method)
        for rank, (document_id, score) in enumerate(ranking, start=1):
            ranking_rows.append({
                "query": query["id"],
                "method": label,
                "rank": rank,
                "document": document_id,
                "score": score,
                "relevant": document_id in query["relevant"],
            })
rankings = pd.DataFrame(ranking_rows)
rankings.to_csv(OUTPUT / "rankings.csv", index=False)
display(rankings[rankings["rank"] <= 3].pivot_table(
    index=["query", "rank"], columns="method", values="document", aggfunc="first"
))


## 8. Evaluation: Precision@3 and MAP

**Precision@3** is the share of the first three results that are relevant:

$$P@3=\frac{\text{relevant results among the top 3}}{3}.$$

**Average Precision (AP)** calculates precision at each relevant result's position, then averages those values. **Mean Average Precision (MAP)** is the mean AP across our five queries. P@3 focuses on the first three results; MAP considers the full ranked list.

For a ranking of relevant, not relevant, relevant: P@3 is $2/3$, and AP is $(1+2/3)/2=0.833$.

These two measures answer different, easy-to-explain questions. We do not need to report several near-duplicate metrics.


In [ ]:
# The shared evaluator returns P@3 and AP for one query.
def evaluate(ranking, relevant, k=3):
    relevant = set(relevant)
    if not relevant:
        raise ValueError("Evaluation requires at least one relevant document.")

    hits = 0
    top_hits = 0
    average_precision = 0.0

    for rank, (document_id, _) in enumerate(ranking, start=1):
        if document_id in relevant:
            hits += 1
            average_precision += hits / rank
            if rank <= k:
                top_hits += 1

    return {
        "P@3": top_hits / k,
        "AP": average_precision / len(relevant),
    }


In [ ]:
metric_rows = []
for query in queries:
    for label, method in methods.items():
        ranking, _ = experiment.rank(query["text"], method=method)
        metric_rows.append({
            "query": query["id"],
            "method": label,
            **evaluate(ranking, query["relevant"]),
        })

metrics = pd.DataFrame(metric_rows)
summary = metrics.groupby("method", sort=False)[["P@3", "AP"]].mean()
summary = summary.rename(columns={"AP": "MAP"})
metrics.to_csv(OUTPUT / "per_query_metrics.csv", index=False)
summary.to_csv(OUTPUT / "summary.csv")
display(summary.round(3))
display(metrics.pivot(index="query", columns="method", values="AP").round(3))



### What the results show

All three methods have mean P@3 of 0.600 on our five queries. Dirichlet query likelihood has MAP 1.000; TF-IDF and BM25 each have MAP 0.900.

On the insurance query, TF-IDF and BM25 rank the parking document D14 first because it repeats `car`. Dirichlet query likelihood ranks the insurance document D13 first because `coverage` also contributes evidence. This does **not** mean smoothing learned that `car` and `automobile` are synonyms.

The results are descriptive only. Five hand-written queries and team-declared labels cannot establish which method is generally better.


## 9. Dirichlet sensitivity

We vary $\mu$ without changing documents or labels. This illustrates how sensitive this small experiment is to the smoothing setting; it is not a held-out tuning procedure. A proper study would choose parameters on development queries and report results on separate test queries.

As $\mu$ approaches zero, the estimates approach MLE. As it becomes very large, document probabilities approach the collection distribution, and score differences shrink.


In [ ]:
sensitivity_rows = []
for mu in [0.1, 1, 5, 20, 100, 1000]:
    values = []
    for query in queries:
        ranking, _ = experiment.rank(query["text"], method="dirichlet", mu=mu)
        values.append(evaluate(ranking, query["relevant"]))
    sensitivity_rows.append({
        "mu": mu,
        "MAP": sum(value["AP"] for value in values) / len(values),
        "P@3": sum(value["P@3"] for value in values) / len(values),
    })

sensitivity = pd.DataFrame(sensitivity_rows)
sensitivity.to_csv(OUTPUT / "sensitivity.csv", index=False)
display(sensitivity.round(3))

lines = ["| $\\mu$ | MAP | P@3 |", "|---:|---:|---:|"]
for row in sensitivity.itertuples():
    lines.append(f"| {row.mu:g} | {row.MAP:.3f} | {row.P_3:.3f} |")
(OUTPUT / "sensitivity-slide.md").write_text("\n".join(lines) + "\n", encoding="utf-8")


## 10. A failure case and unknown words

For `car protection`, our intended target is still the insurance document D13. `protection` is outside the collection, so the declared out-of-vocabulary policy drops it. All three lexical methods then rely on `car` and favor the parking document.

Smoothing gives collection words background support. BM25 balances term frequency and document length. Neither method automatically learns synonym relationships. A fully unknown query returns no ranking because it provides no evidence from this collection.


In [ ]:
for query in collection["diagnostic_queries"]:
    print("\nQuery:", query)
    for label, method in [
        ("TF-IDF", "tfidf"),
        ("Dirichlet", "dirichlet"),
        ("BM25", "bm25"),
    ]:
        ranking, unknown = experiment.rank(query, method=method)
        print(label, "ignored terms:", unknown, "top 3:", ranking[:3])


## 11. Checks for correctness

These checks verify probability normalization, the manual calculation, repeated query terms, consistent tokenization, valid relevance labels, and unknown-query handling. They do not prove that our relevance judgments represent real users.

In [ ]:
for document_id in experiment.document_ids:
    for method in ["mle", "jm", "dirichlet"]:
        total = sum(experiment.term_probability(term, document_id, method)
                    for term in experiment.collection_counts)
        assert math.isclose(total, 1.0, abs_tol=1e-12)
assert tokenize("Language, MODELS!") == ["language", "models"]
assert experiment.rank("quantum teleportation")[0] == []
assert experiment.rank("")[0] == []
one, _ = experiment.rank("language", mu=20)
two, _ = experiment.rank("language language", mu=20)
assert [d for d, _ in one] == [d for d, _ in two]
assert all(math.isclose(dict(two)[d], 2 * score) for d, score in one)
for query in queries:
    assert set(query["relevant"]) <= set(documents)
    for method in ["jm", "dirichlet"]:
        assert all(math.isfinite(score) for _, score in experiment.rank(query["text"], method)[0])
assert math.isclose(summary.loc["TF-IDF", "MAP"], 0.9)
assert math.isclose(summary.loc["Dirichlet (20)", "MAP"], 1.0)
print("All checks passed.")

## 12. Comparison and use in IR

| Method | Main idea | Practical limitation |
|---|---|---|
| Boolean AND | Require every query term | Rejects useful partial matches |
| TF-IDF cosine | Compare weighted term vectors | Depends on exact vocabulary and weighting choices |
| Query likelihood | Estimate query probability under a document model | Needs smoothing and ignores word order in our unigram setup |
| BM25 | Saturate term frequency and normalize by document length | Still mainly lexical and has tunable parameters |

TF-IDF and BM25 are common lexical baselines. Query likelihood offers a probability-based alternative. None of the three automatically understands synonyms or user intent.


## 13. Practice for the defense

1. **Why use unigrams when Lecture 2 teaches bigrams?** Unigrams need fewer counts and suit short queries. Bigrams retain local order but make sparse estimation harder. This is a modeling choice, not a claim that order never matters.
2. **Why do we add logs?** The logarithm preserves ranking and turns products into sums. It also avoids numerical underflow. It does not remove zero probabilities.
3. **Does smoothing make every unknown word possible?** Only terms supported by the chosen background get positive probability. Fully unseen collection terms need a separate policy.
4. **Why is $\mu=20$ not 2500?** Our documents average about 16 words. We use a small illustrative pseudo-count strength and show sensitivity. Real settings need validation.
5. **Does lower perplexity prove better retrieval?** No. Perplexity measures text prediction. We evaluate retrieval using relevance-based metrics.
6. **Did query likelihood beat TF-IDF?** It improved first relevant rank on Q5 and tied on the other four queries. This toy result cannot support a general superiority claim.
7. **Can we compare −6 from one query with −4 from another?** Not as relevance confidence. Query length and term probabilities affect score scale.
8. **What happens if document priors differ?** Add $\log P(d)$ to the query-likelihood log score. We assume equal priors here.

## References

1. DORN Dawin. *Lecture 2: NLP Foundations I: Bag-of-Words and Language Models*. ITC, October 2026. Supplied course PDF, slides 4–8 and 12–20.
2. DORN Dawin. *Lecture 5: From TF-IDF to BM25: Probabilistic Retrieval*. ITC, November 2026. Supplied course PDF, slides 11–34.
3. *Group Discussion Topics: Retrieval Models*. Supplied handout, Topic 2.
4. Manning, C. D., Raghavan, P., & Schütze, H. (2008). *Introduction to Information Retrieval*, Chapters 6 and 12. [BM25 chapter](https://nlp.stanford.edu/IR-book/html/htmledition/okapi-bm25-a-non-binary-model-1.html).
5. Ponte, J. M., & Croft, W. B. (1998). “A Language Modeling Approach to Information Retrieval.” *SIGIR*, 275–281.
6. Robertson, S., & Zaragoza, H. (2009). “The Probabilistic Relevance Framework: BM25 and Beyond.” *Foundations and Trends in Information Retrieval*, 3(4), 333–389. [DOI](https://doi.org/10.1561/1500000019).
